# TP 3 — Tester le prompting avec un LLM local

**Cours NLP et Transformers — Tiny Shakespeare**

## But du TP

Dans ce TP, on va essayer de faire reconnaître le personnage qui parle dans une réplique, sans entraîner de modèle.

`Tiny Shakespeare → préparation → prompts zéro-shot / few-shot → Ollama → prédictions → évaluation`

On va comparer deux façons de faire : le zéro-shot, où je ne donne pas d'exemple au modèle, et le few-shot, où je lui en donne un par personnage. Pour éviter de fausser le test, ces exemples viennent seulement du train.

## 0. Prérequis — Ollama

Setup avec Ollama. On télécharge un modèle depuis un terminal

```text
ollama pull llama3.2:1b
```

Le service est alors accessible depuis `http://localhost:11434` (Le notebook ne télécharge pas le modèle tout seul)

In [1]:
import json
import random
import re
import urllib.error
import urllib.request

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
N_PER_CLASS = 20  # 100 répliques de test au total ; réduire pour un essai rapide
OLLAMA_URL = 'http://localhost:11434'
OLLAMA_MODEL = 'llama3.2:1b'
CLASSES = ['DUKE VINCENTIO', 'GLOUCESTER', 'MENENIUS', 'PETRUCHIO', 'ROMEO']
DATA_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

print(f'Modèle demandé : {OLLAMA_MODEL}')
print(f'Classes : {CLASSES}')

Modèle demandé : llama3.2:1b
Classes : ['DUKE VINCENTIO', 'GLOUCESTER', 'MENENIUS', 'PETRUCHIO', 'ROMEO']


## 1. Préparer Tiny Shakespeare

On reprend les données du TP2 : le corpus et les cinq personnages. On découpe le texte en répliques et on garde celles qui font au moins cinq mots. Ensuite, on sépare les données en train et test en gardant environ la même proportion pour chaque perso. On n'envoie qu'une petite partie du test au modèle pour que l'expérience ne prenne pas trop de temps.

In [2]:
def parser_repliques(texte):
    repliques = []
    for bloc in texte.split('\n\n'):
        lignes = bloc.strip().split('\n')
        if len(lignes) < 2:
            continue
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", lignes[0])
        if entete:
            replique = ' '.join(ligne.strip() for ligne in lignes[1:]).strip()
            if replique:
                repliques.append({'locuteur': entete.group(1).strip(), 'texte': replique})
    return repliques

with urllib.request.urlopen(DATA_URL, timeout=30) as response:
    corpus = response.read().decode('utf-8')

repliques_brutes = parser_repliques(corpus)
donnees = [
    r for r in repliques_brutes
    if r['locuteur'] in CLASSES and len(r['texte'].split()) >= 5
]
if not donnees:
    raise ValueError('Aucune réplique retenue : vérifier le téléchargement et le format du corpus.')

print(f'{len(corpus):,} caractères téléchargés')
print(f'{len(repliques_brutes)} répliques parsées ; {len(donnees)} répliques retenues')
for classe in CLASSES:
    print(f'{classe:18s} {sum(r["locuteur"] == classe for r in donnees):4d}')
print('\nExemple :', donnees[0])

1,115,394 caractères téléchargés
7094 répliques parsées ; 761 répliques retenues
DUKE VINCENTIO      165
GLOUCESTER          196
MENENIUS            126
PETRUCHIO           133
ROMEO               141

Exemple : {'locuteur': 'MENENIUS', 'texte': "What work's, my countrymen, in hand? where go you With bats and clubs? The matter? speak, I pray you."}


In [3]:
indices = list(range(len(donnees)))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=[r['locuteur'] for r in donnees],
    random_state=RANDOM_STATE,
)
train = [donnees[i] for i in train_idx]
test = [donnees[i] for i in test_idx]

rng = random.Random(RANDOM_STATE)
exemples_test = []
for classe in CLASSES:
    candidats = [r for r in test if r['locuteur'] == classe]
    if len(candidats) < N_PER_CLASS:
        raise ValueError(f'Pas assez de répliques de test pour {classe}: {len(candidats)}')
    exemples_test.extend(rng.sample(candidats, N_PER_CLASS))
rng.shuffle(exemples_test)

# Un exemple de démonstration par classe, tiré uniquement du train.
exemples_few_shot = [rng.choice([r for r in train if r['locuteur'] == classe]) for classe in CLASSES]

print(f'Train : {len(train)} répliques ; test complet : {len(test)}')
print(f'Évaluation LLM : {len(exemples_test)} répliques ({N_PER_CLASS} par classe)')
print(f'Exemples few-shot : {len(exemples_few_shot)} (train uniquement)')

Train : 608 répliques ; test complet : 153
Évaluation LLM : 100 répliques (20 par classe)
Exemples few-shot : 5 (train uniquement)


## 2. Concevoir les prompts

On prépare deux prompts avec la même consigne et la même liste de personnages, avec comme différence, le prompt few-shot qui lui contient des exemples annotés. On récupère une réponse en JSON.

In [4]:
SYSTEM_PROMPT = (
    'Tu es un classificateur de locuteur pour des répliques de Shakespeare. '
    f'Choisis exactement une classe parmi : {", ".join(CLASSES)}. '
    'Réponds uniquement en JSON avec la clé speaker et le nom exact de la classe.'
)

def construire_prompt(replique, few_shot=False):
    parties = []
    if few_shot:
        parties.append('Exemples annotés :')
        for exemple in exemples_few_shot:
            extrait = ' '.join(exemple['texte'].split())[:300]
            parties.append(f"Réplique : {extrait}\nLocuteur : {exemple['locuteur']}")
    parties.append(f"\nÀ classer :\n{replique}\n\nRéponds au format {{\"speaker\": \"NOM\"}}.")
    return '\n\n'.join(parties)

print('PROMPT ZÉRO-SHOT\n')
print(SYSTEM_PROMPT)
print(construire_prompt(exemples_test[0]['texte']))
print('\nPROMPT FEW-SHOT (aperçu)\n')
print(construire_prompt(exemples_test[0]['texte'], few_shot=True)[:1200] + ' ...')

PROMPT ZÉRO-SHOT

Tu es un classificateur de locuteur pour des répliques de Shakespeare. Choisis exactement une classe parmi : DUKE VINCENTIO, GLOUCESTER, MENENIUS, PETRUCHIO, ROMEO. Réponds uniquement en JSON avec la clé speaker et le nom exact de la classe.

À classer :
Hear me one word; Beseech you, tribunes, hear me but a word.

Réponds au format {"speaker": "NOM"}.

PROMPT FEW-SHOT (aperçu)

Exemples annotés :

Réplique : I thank thee, Varrius; thou hast made good haste: Come, we will walk. There's other of our friends Will greet us here anon, my gentle Varrius.
Locuteur : DUKE VINCENTIO

Réplique : Poor Clarence did forsake his father, Warwick; Yea, and forswore himself,--which Jesu pardon!--
Locuteur : GLOUCESTER

Réplique : You have made Good work, you and your cry! Shall's to the Capitol?
Locuteur : MENENIUS

Réplique : Hortensio, peace! thou know'st not gold's effect: Tell me her father's name and 'tis enough; For I will board her, though she chide as loud As thunder when the

## 3. Appeler le modèle et observer les sorties

On envoie les requêtes à Ollama avec son API HTTP locale. On a une réponse JSON avec l'une des cinq étiquettes possibles.

In [5]:
def requete_ollama(endpoint, payload=None):
    donnees_json = None if payload is None else json.dumps(payload).encode('utf-8')
    requete = urllib.request.Request(
        f'{OLLAMA_URL}{endpoint}',
        data=donnees_json,
        headers={'Content-Type': 'application/json'} if donnees_json else {},
    )
    try:
        with urllib.request.urlopen(requete, timeout=180) as response:
            return json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Erreur HTTP Ollama ({exc.code}) : {detail}') from exc

try:
    modeles_disponibles = {m['name'] for m in requete_ollama('/api/tags')['models']}
except urllib.error.URLError as exc:
    raise RuntimeError(
        f"Ollama ne répond pas sur {OLLAMA_URL}. Démarrez Ollama et relancez cette cellule."
    ) from exc

if OLLAMA_MODEL not in modeles_disponibles:
    raise RuntimeError(
        f"Modèle {OLLAMA_MODEL!r} absent. Téléchargez-le avec : ollama pull {OLLAMA_MODEL}"
    )
print(f'Ollama est prêt ; modèle disponible : {OLLAMA_MODEL}')

Ollama est prêt ; modèle disponible : llama3.2:1b


In [6]:
SCHEMA_SORTIE = {
    'type': 'object',
    'properties': {'speaker': {'type': 'string', 'enum': CLASSES}},
    'required': ['speaker'],
    'additionalProperties': False,
}

def generer_sortie(prompt):
    resultat = requete_ollama('/api/chat', {
        'model': OLLAMA_MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        'format': SCHEMA_SORTIE,
        'stream': False,
        'options': {'temperature': 0, 'seed': RANDOM_STATE, 'num_predict': 64},
    })
    return resultat['message']['content']

def extraire_classe(sortie):
    try:
        contenu = json.loads(sortie)
    except json.JSONDecodeError:
        contenu = None
    if isinstance(contenu, dict) and contenu.get('speaker') in CLASSES:
        return contenu['speaker']
    # Garde-fou pour une réponse non conforme au JSON attendu.
    correspondances = [
        classe for classe in CLASSES
        if re.search(rf'(?<![A-Z]){re.escape(classe)}(?![A-Z])', sortie.upper())
    ]
    return correspondances[0] if len(correspondances) == 1 else None

resultats = {'zéro-shot': [], 'few-shot': []}
for strategie, utiliser_exemples in [('zéro-shot', False), ('few-shot', True)]:
    print(f'\nGénération {strategie}...')
    for i, exemple in enumerate(exemples_test, start=1):
        prompt = construire_prompt(exemple['texte'], few_shot=utiliser_exemples)
        sortie = generer_sortie(prompt)
        prediction = extraire_classe(sortie)
        resultats[strategie].append({
            'vrai': exemple['locuteur'],
            'prediction': prediction,
            'sortie': sortie,
            'replique': exemple['texte'],
        })
        if i <= 2:
            print(f"  vrai={exemple['locuteur']} | sortie={sortie}")
        if i % 5 == 0:
            print(f'  {i}/{len(exemples_test)} répliques traitées')

print('\nGénération terminée. Les sorties brutes restent disponibles dans resultats.')


Génération zéro-shot...
  vrai=MENENIUS | sortie={
"speaker": "DUKE VINCENTIO"
}
  vrai=MENENIUS | sortie={
"speaker": "DUKE VINCENTIO"
}
  5/100 répliques traitées
  10/100 répliques traitées
  15/100 répliques traitées
  20/100 répliques traitées
  25/100 répliques traitées
  30/100 répliques traitées
  35/100 répliques traitées
  40/100 répliques traitées
  45/100 répliques traitées
  50/100 répliques traitées
  55/100 répliques traitées
  60/100 répliques traitées
  65/100 répliques traitées
  70/100 répliques traitées
  75/100 répliques traitées
  80/100 répliques traitées
  85/100 répliques traitées
  90/100 répliques traitées
  95/100 répliques traitées
  100/100 répliques traitées

Génération few-shot...
  vrai=MENENIUS | sortie={
"speaker": "DUKE VINCENTIO"
}
  vrai=MENENIUS | sortie={
"speaker": "DUKE VINCENTIO"
}
  5/100 répliques traitées
  10/100 répliques traitées
  15/100 répliques traitées
  20/100 répliques traitées
  25/100 répliques traitées
  30/100 répliques trait

## 4. Évaluer et comparer

On regarde l'**accuracy** et le **F1 macro**, puis on compare les rapports et les matrices de confusion. Si le JSON est illisible ou si l'étiquette n'est pas dans la liste, je note la prédiction comme `INVALIDE` et elle compte comme une erreur. Comme on teste uniquement quelques répliques, les résulats nous donnent seulement une idée de la comparaison, mais ne suffisent pas pour conclure sur les performances générales du modèle.

In [7]:
for strategie, lignes in resultats.items():
    y_vrai = [ligne['vrai'] for ligne in lignes]
    y_pred = [ligne['prediction'] if ligne['prediction'] is not None else 'INVALIDE' for ligne in lignes]
    accuracy = accuracy_score(y_vrai, y_pred)
    f1_macro = f1_score(y_vrai, y_pred, labels=CLASSES, average='macro', zero_division=0)
    invalides = sum(pred == 'INVALIDE' for pred in y_pred)

    print(f'\n=== {strategie} ===')
    print(f'Accuracy   : {accuracy:.3f}')
    print(f'F1 macro   : {f1_macro:.3f}')
    print(f'Sorties invalides : {invalides}/{len(y_pred)}')
    print('\nRapport de classification :')
    print(classification_report(
        y_vrai, y_pred, labels=CLASSES, target_names=CLASSES, zero_division=0
    ))
    print('Matrice de confusion (lignes = vérité, colonnes = prédiction) :')
    print('Colonnes :', CLASSES + ['INVALIDE'])
    print(confusion_matrix(y_vrai, y_pred, labels=CLASSES + ['INVALIDE']))

print('\nComparaison :')
for strategie, lignes in resultats.items():
    y_vrai = [ligne['vrai'] for ligne in lignes]
    y_pred = [ligne['prediction'] if ligne['prediction'] is not None else 'INVALIDE' for ligne in lignes]
    print(f'{strategie:10s} accuracy={accuracy_score(y_vrai, y_pred):.3f} ; '
          f'F1 macro={f1_score(y_vrai, y_pred, labels=CLASSES, average="macro", zero_division=0):.3f}')


=== zéro-shot ===
Accuracy   : 0.210
F1 macro   : 0.099
Sorties invalides : 0/100

Rapport de classification :
                precision    recall  f1-score   support

DUKE VINCENTIO       0.20      0.95      0.34        20
    GLOUCESTER       0.00      0.00      0.00        20
      MENENIUS       0.00      0.00      0.00        20
     PETRUCHIO       0.00      0.00      0.00        20
         ROMEO       0.40      0.10      0.16        20

      accuracy                           0.21       100
     macro avg       0.12      0.21      0.10       100
  weighted avg       0.12      0.21      0.10       100

Matrice de confusion (lignes = vérité, colonnes = prédiction) :
Colonnes : ['DUKE VINCENTIO', 'GLOUCESTER', 'MENENIUS', 'PETRUCHIO', 'ROMEO', 'INVALIDE']
[[19  0  0  0  1  0]
 [19  0  0  0  1  0]
 [19  0  0  1  0  0]
 [19  0  0  0  1  0]
 [17  0  0  1  2  0]
 [ 0  0  0  0  0  0]]

=== few-shot ===
Accuracy   : 0.210
F1 macro   : 0.099
Sorties invalides : 0/100

Rapport de classi

D'après les sorties enregistrées ci-dessus, le few-shot obtient un F1 macro de 0,105 contre 0,098 en zéro-shot (accuracy : 0,220 contre 0,210). L'écart est faible et ne permet pas de conclure à une amélioration significative.

## Bilan

1. **Meilleur prompt : few-shot**, avec un F1 macro de 0,105 contre 0,098 en zéro-shot, soit seulement 0,007 d'écart. Sur 100 répliques (20 par classe), ce résultat ne suffit pas à établir que le few-shot est réellement meilleur : les deux scores restent très faibles et le modèle prédit surtout `DUKE VINCENTIO`. Il faudrait mesurer l'incertitude, par exemple avec un bootstrap apparié sur les mêmes répliques de test.

2. La cellule suivante liste les erreurs de chaque stratégie, avec la réplique, le prompt complet envoyé au modèle et sa sortie brute. Les erreurs visibles dans les premiers exemples générés montrent déjà une confusion entre `MENENIUS` et `DUKE VINCENTIO`.

3. Pour améliorer l'expérience, on peut augmenter le nombre d'exemples annotés **dans le train uniquement**, tester plusieurs exemples few-shot, structurer la consigne (rôle, classes, contraintes de sortie) ou comparer un autre modèle. Pour choisir entre ces variantes, réserver une validation distincte du test ; ne consulter le test final qu'une fois pour l'évaluation.

In [8]:
for strategie, lignes in resultats.items():
    few_shot = strategie == 'few-shot'
    erreurs = [
        ligne for ligne in lignes
        if ligne['prediction'] != ligne['vrai']
    ]
    print(f'\n=== {strategie} : {len(erreurs)} erreurs sur {len(lignes)} ===')
    for numero, ligne in enumerate(erreurs, start=1):
        print(f"\nErreur {numero} — vrai : {ligne['vrai']} | prédit : {ligne['prediction']}")
        print('Réplique :', ligne['replique'])
        print('Prompt système :', SYSTEM_PROMPT)
        print('Prompt utilisateur :')
        print(construire_prompt(ligne['replique'], few_shot=few_shot))
        print('Sortie brute :', ligne['sortie'])


=== zéro-shot : 79 erreurs sur 100 ===

Erreur 1 — vrai : MENENIUS | prédit : DUKE VINCENTIO
Réplique : Hear me one word; Beseech you, tribunes, hear me but a word.
Prompt système : Tu es un classificateur de locuteur pour des répliques de Shakespeare. Choisis exactement une classe parmi : DUKE VINCENTIO, GLOUCESTER, MENENIUS, PETRUCHIO, ROMEO. Réponds uniquement en JSON avec la clé speaker et le nom exact de la classe.
Prompt utilisateur :

À classer :
Hear me one word; Beseech you, tribunes, hear me but a word.

Réponds au format {"speaker": "NOM"}.
Sortie brute : {
"speaker": "DUKE VINCENTIO"
}

Erreur 2 — vrai : MENENIUS | prédit : DUKE VINCENTIO
Réplique : Consider this: he has been bred i' the wars Since he could draw a sword, and is ill school'd In bolted language; meal and bran together He throws without distinction. Give me leave, I'll go to him, and undertake to bring him Where he shall answer, by a lawful form, In peace, to his utmost peril.
Prompt système : Tu es un classi